In [1]:
import numpy as np
import matplotlib.pyplot as plt
import random
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from scipy.stats import ortho_group

# Repro seeds (adjust as you like)
np.random.seed(0)
random.seed(0)
torch.manual_seed(0)
torch.backends.cudnn.enabled = False

print(torch.__version__)



#bias & variance 打印。。。调整IPW performance;

####真实实验
import pandas as pd
import numpy as np


# 1) 读取数据（Vanderbilt CSV）
url = "https://hbiostat.org/data/repo/rhc.csv"  # 官方公开地址

# /Users/zhangzhiheng/Downloads/rhc.csv
# url = "/Users/zhangzhiheng/Downloads/rhc.csv"

df = pd.read_csv(url)

# 2) 基本清洗：统一大小写/空白、去掉明显非建模字段
drop_cols = ["ptid","sadmdte","dschdte","dthdte","lstctdte","t3d30"]  # 日期/ID/跟踪日期等
for c in drop_cols:
    if c in df.columns: df.drop(columns=c, inplace=True)

# 3) 构造 A / Y
A = (df["swang1"].astype(str).str.strip().str.upper() == "RHC").astype(np.float32).values[:,None]

# 30天生存（与表格方向一致：若用死亡则取 1{dth30=="Yes"} 并注意号）
Y = (df["dth30"].astype(str).str.strip().str.upper() == "NO").astype(np.float32).values[:,None]

# 4) 负控变量
Z_cols = ["pafi1","paco21"]
W_cols = ["ph1","hema1"]

# 确保都存在
for c in Z_cols+W_cols:
    assert c in df.columns, f"缺少列 {c}"

# 5) X = 其余基线协变量（去掉 A/Y/W/Z）
use_df = df.copy()
use_df = use_df.drop(columns=["swang1","dth30"])             # 原始的 A/Y 字段
use_df = use_df.drop(columns=Z_cols+W_cols)

# 剔除明显不可用的列（如全空、唯一值列）
use_df = use_df.loc[:, use_df.apply(lambda s: (~s.isna()).sum()>0)]
use_df = use_df.loc[:, use_df.nunique(dropna=True) > 1]

# 6) 数值化：类别做 one-hot，连续标准化；W/Z、X 分开处理
cont = use_df.select_dtypes(include=[np.number]).columns.tolist()
cat  = [c for c in use_df.columns if c not in cont]

X_mat = pd.get_dummies(use_df, columns=cat, drop_first=True)
# 简单均值填补
X_mat = X_mat.fillna(X_mat.mean())
# 标准化
X_mat = (X_mat - X_mat.mean())/X_mat.std(ddof=0)

Z_mat = df[Z_cols].astype(np.float32).values
W_mat = df[W_cols].astype(np.float32).values

# 对 Z/W 也做简单填补+标准化
for M in (Z_mat, W_mat):
    # 均值填补
    col_means = np.nanmean(M, axis=0)
    inds = np.where(np.isnan(M))
    M[inds] = np.take(col_means, inds[1])
Z_mat = (Z_mat - Z_mat.mean(0))/Z_mat.std(0, ddof=0)
W_mat = (W_mat - W_mat.mean(0))/W_mat.std(0, ddof=0)


# 7) 转成你现有代码需要的名字/形状
x_list = X_mat.values.astype(np.float32)                   # (n, d_x)
z_list = Z_mat.astype(np.float32)                          # (n, 2)
w_list = W_mat.astype(np.float32)                          # (n, 2)
a_list = A.astype(np.float32)                              # (n, 1)
y_list = Y.astype(np.float32)                              # (n, 1)

print(x_list.shape, z_list.shape, w_list.shape, a_list.shape, y_list.shape)

















2.8.0
(5735, 70) (5735, 2) (5735, 2) (5735, 1) (5735, 1)


In [2]:
# ==========================================
# Proximal Minimax DR vs. DR(sta) on RHC data
# - 核损失里做 1{A=a} 掩码
# - 稳定子通过 K ← K + lam_k * I
# - DR:     train_e(lam_k=0.0)
# - DR(sta):train_e(lam_k=1.0)
# ==========================================
import numpy as np, pandas as pd
import torch, torch.nn as nn, torch.nn.functional as F, torch.optim as optim
from math import sqrt

# ---------------- 1) Load & preprocess ----------------
url = "https://hbiostat.org/data/repo/rhc.csv"   # Vanderbilt Biostatistics RHC
df = pd.read_csv(url)

# A = RHC 指示；Y = 30-day survival (和论文表方向一致)
A = (df["swang1"].astype(str).str.upper().str.strip() == "RHC").astype(np.float32).values[:, None]
Y = (df["dth30"].astype(str).str.upper().str.strip() == "NO").astype(np.float32).values[:, None]

# 负控变量（与论文设置一致）
Z_cols = ["pafi1", "paco21"]   # NC exposures
W_cols = ["ph1",   "hema1"]    # NC outcomes





from itertools import combinations
# 所有变量
all_cols = ["pafi1", "paco21", "ph1", "hema1"]
# 生成所有 6 种分配方案：从 4 个里选 2 个做 Z，其余做 W
schemes = []
for z_pair in combinations(all_cols, 2):
    w_pair = [col for col in all_cols if col not in z_pair]
    schemes.append((list(z_pair), w_pair))
# === 这里用一个参数来控制采用哪一种方案（1~6）===
scheme_id = 4   # 改成 1,2,3,4,5,6 之一
if not (1 <= scheme_id <= len(schemes)):
    raise ValueError(f"scheme_id 必须在 1~{len(schemes)} 之间")
Z_cols, W_cols = schemes[scheme_id - 1]
print(f"当前使用的方案 {scheme_id}:")
print("Z_cols =", Z_cols)
print("W_cols =", W_cols)





for c in Z_cols + W_cols:
    if c not in df.columns:
        raise ValueError(f"Missing column {c} in RHC csv")

# X = baseline 协变量（除去 A、Y、W、Z 及明显非特征列）
drop_cols = ["ptid","sadmdte","dschdte","dthdte","lstctdte","t3d30","swang1","dth30"] + Z_cols + W_cols
X_df = df.drop(columns=[c for c in drop_cols if c in df.columns], errors="ignore")

# One-hot（drop_first 以避免共线），均值填补 + 标准化
num_cols = X_df.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = [c for c in X_df.columns if c not in num_cols]
X = pd.get_dummies(X_df, columns=cat_cols, drop_first=True)
X = X.replace([np.inf, -np.inf], np.nan).fillna(X.mean()).astype(np.float32)
X = ((X - X.mean())/X.std(ddof=0)).fillna(0.0)

# W, Z 简单均值填补 + 标准化
def stdize(M):
    M = M.astype(np.float32).copy()
    m = np.nanmean(M, axis=0); M[np.isnan(M)] = np.take(m, np.where(np.isnan(M))[1])
    M -= M.mean(0); M /= (M.std(0, ddof=0) + 1e-8)
    return M

Z = stdize(df[Z_cols].values)
W = stdize(df[W_cols].values)

X = X.values.astype(np.float32)
A = A.astype(np.float32); Y = Y.astype(np.float32)

n, dx = X.shape
dz, dw = Z.shape[1], W.shape[1]
print(f"Loaded RHC: n={n}, dimX={dx}, dimZ={dz}, dimW={dw}")

# torch tensors
x_all = torch.from_numpy(X).float()
z_all = torch.from_numpy(Z).float()
w_all = torch.from_numpy(W).float()
a_all = torch.from_numpy(A).float()
y_all = torch.from_numpy(Y).float()

# ---------------- 2) Kernels ----------------
def median_heuristic_sq(X_t):
    with torch.no_grad():
        D2 = torch.cdist(X_t, X_t, p=2)**2
        triu = D2[torch.triu(torch.ones_like(D2), diagonal=1)==1]
        if torch.any(triu>0): m = torch.median(triu[triu>0])
        else: m = torch.tensor(1.0, device=X_t.device)
    return float(m.item())

def rbf_gram(X_t, gamma=None):
    # 注意：不在这里添加岭化；稳定子在训练里作为 K += lam_k*I 实现
    D2 = torch.cdist(X_t, X_t, p=2)**2
    if gamma is None:
        m = median_heuristic_sq(X_t)
        gamma = 0.5 / max(m, 1e-12)
    K = torch.exp(-gamma * D2)
    return K

# ---------------- 3) Nets ----------------
# class HNet(nn.Module):  # h_a(W,X)
#     def __init__(self, d_w, d_x, h=256):
#         super().__init__()
#         self.fc1 = nn.Linear(d_w + d_x, h)
#         self.fc2 = nn.Linear(h, h//2)
#         self.fc3 = nn.Linear(h//2, 1)
#     def forward(self, wx):
#         x = F.relu(self.fc1(wx))
#         x = F.relu(self.fc2(x))
#         return self.fc3(x)


# class HNet(nn.Module):  # h_a(W,X)
#     def __init__(self, d_w, d_x, h=256):
#         super().__init__()
#         self.fc1  = nn.Linear(d_w + d_x, h)
#         self.fc2  = nn.Linear(h, h//2)
#         self.fc2b = nn.Linear(h//2, h//2)   # ← 新增的隐藏层
#         self.fc3  = nn.Linear(h//2, 1)      # 原来的输出层不改

#     def forward(self, wx):
#         x = F.relu(self.fc1(wx))
#         x = F.relu(self.fc2(x))
#         x = F.relu(self.fc2b(x))            # ← 新增一次 ReLU
#         return self.fc3(x)

class HNet(nn.Module):
    def __init__(self, d_w, d_x, h=256):
        super().__init__()
        self.fc1 = nn.Linear(d_w + d_x, h)
        self.fc2 = nn.Linear(h, h//2)
        self.fc3 = nn.Linear(h//2, h//2)  # 新的隐藏层
        self.fc4 = nn.Linear(h//2, 1)     # 新的输出层

    def forward(self, wx):
        x = F.relu(self.fc1(wx))
        x = F.relu(self.fc2(x))
        x = F.relu(self.fc3(x))
        return self.fc4(x)

import torch.nn as nn
import torch.nn.functional as F

# class HNet(nn.Module):  # h_a(W, X)
#     def __init__(self, d_w, d_x, h=256):
#         super().__init__()
#         self.fc1 = nn.Linear(d_w + d_x, h)      # 隐藏层1
#         self.fc2 = nn.Linear(h, h // 2)         # 隐藏层2
#         self.fc3 = nn.Linear(h // 2, h // 2)    # 隐藏层3（新增）
#         self.fc4 = nn.Linear(h // 2, h // 2)    # 隐藏层4（新增）
#         self.fc5 = nn.Linear(h // 2, 1)         # 输出层（原 fc3 后移）

#     def forward(self, wx):
#         x = F.relu(self.fc1(wx))
#         x = F.relu(self.fc2(x))
#         x = F.relu(self.fc3(x))
#         x = F.relu(self.fc4(x))
#         return self.fc5(x)



# class ENet(nn.Module):  # e(Z,X)=P(A=1|Z,X)
#     def __init__(self, d_z, d_x, h=256):
#         super().__init__()
#         self.fc1 = nn.Linear(d_z + d_x, h)
#         self.fc2 = nn.Linear(h, h//2)
#         self.fc3 = nn.Linear(h//2, 1)
#     def forward(self, zx):
#         x = F.relu(self.fc1(zx))
#         x = F.relu(self.fc2(x))
#         return torch.sigmoid(self.fc3(x))



class ENet(nn.Module):  # e(Z,X)=P(A=1|Z,X)
    def __init__(self, d_z, d_x, h=256):
        super().__init__()
        self.fc1 = nn.Linear(d_z + d_x, h)
        self.fc2 = nn.Linear(h, h//2)
        self.fc3 = nn.Linear(h//2, h//2)
        self.fc4 = nn.Linear(h // 2, 1) 
        
    def forward(self, zx):
        x = F.relu(self.fc1(zx))
        x = F.relu(self.fc2(x))
        x = F.relu(self.fc3(x))
        return torch.sigmoid(self.fc4(x))


#class HNET   1/p ~ q();    E(Y) ~ h()


# ---------------- 4) Trainers（按你的“正确流程”实现） ----------------
def train_h(network, optimizer, X, W, Z, Y, A, batch, n_epochs, lam_k=0.1, center_res=True, a_value=1.0):
    """
    Outcome bridge for arm a:  minimize r^T (K_{ZX} + lam_k I) r / b^2
    r = 1{A=a} * (Y - h(W,X));  核输入是 [Z,X]
    """
    n = X.size(0)
    losses = []
    Ibuf = None
    steps = max(1, n // batch)
    for epoch in range(1, n_epochs+1):
        perm = torch.randperm(n)
        epoch_loss = 0.0
        for i in range(steps):
            idx = perm[i*batch : min((i+1)*batch, n)]
            x_b, w_b, z_b, y_b, a_b = X[idx], W[idx], Z[idx], Y[idx], A[idx]
            wx_b = torch.cat([w_b, x_b], dim=1)      # h input
            zx_b = torch.cat([z_b, x_b], dim=1)      # critic kernel input

            pred = network(wx_b)                      # h(W,X)
            r = (y_b - pred) * (a_b == a_value).float()  # 掩码：1{A=a}·(Y - h)

            if center_res:
                r = r - r.mean()

            K = rbf_gram(zx_b)                        # K_{ZX}
            bsz = K.size(0)
            if Ibuf is None or Ibuf.size(0) != bsz:
                Ibuf = torch.eye(bsz, device=K.device)
            K = K + lam_k * Ibuf                      # 稳定子：岭化在 Gram 上

            loss = (r.t() @ K @ r) / (bsz**2)

            optimizer.zero_grad(); loss.backward(); optimizer.step()
            epoch_loss += float(loss.item())
        losses.append(epoch_loss / steps)
    return losses

def train_e(network, optimizer, X, W, Z, A, batch, n_epochs, lam_k=0.1, center_res=True):
    """
    Treatment bridge: minimize r^T (K_{WX} + lam_k I) r / b^2
    r = 1{A=1} * (A - e(Z,X));  核输入是 [W,X]
    """
    n = X.size(0)
    losses = []
    Ibuf = None
    steps = max(1, n // batch)
    for epoch in range(1, n_epochs+1):
        perm = torch.randperm(n)
        epoch_loss = 0.0
        for i in range(steps):
            idx = perm[i*batch : min((i+1)*batch, n)]
            x_b, w_b, z_b, a_b = X[idx], W[idx], Z[idx], A[idx]
            zx_b = torch.cat([z_b, x_b], dim=1)      # e input
            wx_b = torch.cat([w_b, x_b], dim=1)      # critic kernel input

            ehat = network(zx_b)                      # e(Z,X)
            # r = (a_b - ehat) * (a_b == 1).float()     # 此处我做了修改
            r = (a_b - ehat) 
            
            if center_res:
                r = r - r.mean()

            K = rbf_gram(wx_b)                        # K_{WX}
            bsz = K.size(0)
            if Ibuf is None or Ibuf.size(0) != bsz:
                Ibuf = torch.eye(bsz, device=K.device)
            K = K + lam_k * Ibuf                      # 稳定子：岭化在 Gram 上

            loss = (r.t() @ K @ r) / (bsz**2)

            optimizer.zero_grad(); loss.backward(); optimizer.step()
            epoch_loss += float(loss.item())
        losses.append(epoch_loss / steps)
    return losses
def train_h_sta(network, optimizer, X, W, Z, Y, A,
                       batch, n_epochs, lam_k=0.01, gamma1=1, center_res=True, a_value=1):
    """
    Stabilized outcome bridge training (Lemma 34):
        minimize ψᵀ K (γ₁I + λK)⁻¹ ψ / n²
    ψ_i = 1{A=1}(Y_i - h(W_i,1,X_i))
    """
    n = X.size(0)
    losses = []
    Ibuf = None
    for epoch in range(1, n_epochs+1):
        perm = torch.randperm(n)
        epoch_loss = 0.0
        for i in range(max(1, n // batch)):
            idx = perm[i*batch : min((i+1)*batch, n)]
            x_b, w_b, z_b, y_b, a_b = X[idx], W[idx], Z[idx], Y[idx], A[idx]

            wx_b = torch.cat([w_b, x_b], dim=1)
            zx_b = torch.cat([z_b, x_b], dim=1)

            pred = network(wx_b)
            r = (y_b - pred) * (a_b == a_value).float()
            if center_res:
                r = r - r.mean()

            K = rbf_gram(zx_b)
            bsz = K.size(0)
            device = K.device
            if Ibuf is None or Ibuf.size(0) != bsz:
                Ibuf = torch.eye(bsz, device=device)

            # (γ₁I + λK) inverse
            stabilizer = gamma1 * Ibuf + lam_k * K
            inv_stab = torch.linalg.solve(stabilizer, Ibuf)  # safe inverse
            loss = (r.t() @ (K @ inv_stab @ r)) / (bsz**2)
            loss = loss.squeeze()


            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            epoch_loss += float(loss.item())
        losses.append(epoch_loss / max(1, n // batch))
    return losses

def train_e_sta(network, optimizer,
                   X, W, Z, A,
                   batch, n_epochs,
                   lam_k=0.1,        # λ
                   gamma2=1,       # γ₂
                   center_res=False,
                   monitor_every=5):
    """
    Stabilized q-bridge training, corresponding to Eq. (42):

        q_hat = argmin_q  φ(q)^T K_{w1}^{1/2} (λK_{w1}+γ₂I)^{-1} K_{w1}^{1/2} φ(q)
                         - 2 φ(q)^T (λK_{w1}+γ₂I)^{-1} K_{w2} 1_n.

    在 Example 1 下，我们对 target action a0=1 做近似：
      - 仅使用 {i: A_i=1} 子样本
      - φ_i(q) = q(Z_i, 1, X_i)
      - K_{w1} = Gram( (W_i,X_i) )
      - K_{w2} ≈ K_{w1}

    并利用恒等式：
      K_{w1}^{1/2} (λK_{w1}+γ₂I)^{-1} K_{w1}^{1/2}
      = K_{w1} (λK_{w1}+γ₂I)^{-1}
    避免显式计算 K^{1/2}.
    """

    n = X.size(0)
    device = X.device
    losses = []
    Ibuf = None

    for epoch in range(1, n_epochs + 1):
        perm = torch.randperm(n)
        epoch_loss = 0.0

        for i in range(max(1, n // batch)):
            idx = perm[i * batch : min((i + 1) * batch, n)]
            x_b = X[idx]
            w_b = W[idx]
            z_b = Z[idx]
            a_b = A[idx].view(-1, 1)

            # 只用 A=1 的样本，等价于 Example 1 的 product kernel
            mask1 = (a_b <= 1 ).view(-1)
            if mask1.sum() < 2:
                continue

            X1 = x_b[mask1]
            W1 = w_b[mask1]
            Z1 = z_b[mask1]

            ZX1 = torch.cat([Z1, X1], dim=1)   # q 输入
            WX1 = torch.cat([W1, X1], dim=1)   # kernel 输入
            b1 = ZX1.size(0)

            # φ_n(q) = q(Z,1,X)
            qhat = network(ZX1)
            if qhat.dim() == 1:
                qhat = qhat.view(-1, 1)
            assert qhat.shape == (b1, 1), f"qhat shape {qhat.shape}, expected ({b1},1)"

            phi = qhat
            if center_res:
                phi = phi - phi.mean()
            phi = a_b * 1/(phi.clamp_(0.05, 0.95))

            # K_{w1}, K_{w2}
            K1 = rbf_gram(WX1)           # (b1, b1)
            K2 = K1                      # Example 1 近似：K_{w2} = K_{w1}

            if (Ibuf is None) or (Ibuf.size(0) != b1):
                Ibuf = torch.eye(b1, device=device)

            # S = λK_{w1} + γ₂ I
            S = lam_k * K1 + gamma2 * Ibuf

            # M = (λK_{w1} + γ₂ I)^{-1} K_{w1}
            # (与 K_{w1}^{1/2}(λK_{w1}+γ₂I)^{-1}K_{w1}^{1/2} 等价)
            M = torch.linalg.solve(S, K1)       # shape (b1,b1)

            # 第一项： φ^T M φ
            term1 = (phi.t() @ (M @ phi))[0, 0]

            # 第二项：2 φ^T (λK_{w1}+γ₂I)^{-1} K_{w2} 1
            ones = torch.ones(b1, 1, device=device)
            b_vec = torch.linalg.solve(S, K2 @ ones)  # shape (b1,1)
            term2 = (2.0 * (phi.t() @ b_vec))[0, 0]

            loss = (term1 - term2) / (b1 ** 2)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            epoch_loss += loss.item()

        losses.append(epoch_loss / max(1, n // batch))

        if (monitor_every is not None) and (epoch % monitor_every == 0):
            print(f"[train_e_stable] epoch {epoch}, loss={losses[-1]:.6g}")

    return losses







# ---------------- 5) 点估计 ----------------
torch.manual_seed(0); np.random.seed(0)
batch = 256
epochs_point = 500
clamp_eps = 0.05   # 避免极端比值
lam_h = 1.0        # h 的稳定子固定使用 1.0（按你们之前实现）

# 训练 h1, h0（各自 arm 上掩码）
H1 = HNet(dw, dx); optH1 = optim.Adam(H1.parameters(), lr=1e-3)
H0 = HNet(dw, dx); optH0 = optim.Adam(H0.parameters(), lr=1e-3)
_ = train_h(H1, optH1, x_all, w_all, z_all, y_all, a_all, batch, n_epochs=epochs_point, lam_k=lam_h, center_res=True, a_value=1.0)
_ = train_h(H0, optH0, x_all, w_all, z_all, y_all, a_all, batch, n_epochs=epochs_point, lam_k=lam_h, center_res=True, a_value=0.0)


##add##############################################################
H1_sta = HNet(dw, dx); optH1 = optim.Adam(H1_sta.parameters(), lr=1e-3)
H0_sta = HNet(dw, dx); optH0 = optim.Adam(H0_sta.parameters(), lr=1e-3)
_ = train_h_sta(H1_sta, optH1, x_all, w_all, z_all, y_all, a_all, batch, n_epochs=epochs_point, lam_k=lam_h, center_res=True, a_value=1.0)
_ = train_h_sta(H0_sta, optH0, x_all, w_all, z_all, y_all, a_all, batch, n_epochs=epochs_point, lam_k=lam_h, center_res=True, a_value=0.0)
##add##############################################################


# e: 两个版本 —— DR 用 lam_k=0.0；DR(sta) 用 lam_k=1.0
E_dr  = ENet(dz, dx); optE_dr  = optim.Adam(E_dr.parameters(),  lr=1e-3)
_ = train_e(E_dr,  optE_dr,  x_all, w_all, z_all, a_all, batch, n_epochs=epochs_point, lam_k=0.0, center_res=True)



E_sta = ENet(dz, dx); optE_sta = optim.Adam(E_sta.parameters(), lr=1e-3)
_ = train_e_sta(E_sta, optE_sta, x_all, w_all, z_all, a_all, batch, n_epochs=epochs_point, lam_k=5.0,gamma2=100, center_res=True, monitor_every=5)


    

    
with torch.no_grad():
    wx = torch.cat([w_all, x_all], dim=1)
    zx = torch.cat([z_all, x_all], dim=1)
    h1 = H1(wx); h0 = H0(wx)
    h1_sta = H1_sta(wx); h0_sta = H0_sta(wx)
    e_dr  = E_dr(zx).clamp_(clamp_eps, 1.0-clamp_eps)
    e_sta = E_sta(zx).clamp_(clamp_eps, 1.0-clamp_eps)

    wA_dr,  w0A_dr  = a_all/e_dr,  (1.0-a_all)/(1.0-e_dr)
    wA_st,  w0A_st  = a_all/e_sta, (1.0-a_all)/(1.0-e_sta)

    DR_point     = torch.mean(h1 - h0 + wA_dr*(y_all - h1) - w0A_dr*(y_all - h0)).item()
    DR_sta_point = torch.mean(h1_sta - h0_sta + wA_st*(y_all - h1_sta) - w0A_st*(y_all - h0_sta)).item()

print(f"Point estimates -> DR: {DR_point:+.6f} | DR(sta): {DR_sta_point:+.6f}")

# # ---------------- 6) Bootstrap SEs & 95% CIs（paired bootstrap） ----------------
# def fit_and_estimate(X, Z, W, A, Y, epochs=500):
#     X_t = torch.from_numpy(X).float()
#     Z_t = torch.from_numpy(Z).float()
#     W_t = torch.from_numpy(W).float()
#     A_t = torch.from_numpy(A).float()
#     Y_t = torch.from_numpy(Y).float()

#     H1 = HNet(dw, dx); optH1 = optim.Adam(H1.parameters(), lr=1e-3)
#     H0 = HNet(dw, dx); optH0 = optim.Adam(H0.parameters(), lr=1e-3)
#     train_h(H1, optH1, X_t, W_t, Z_t, Y_t, A_t, batch, n_epochs=epochs, lam_k=lam_h, center_res=True, a_value=1.0)
#     train_h(H0, optH0, X_t, W_t, Z_t, Y_t, A_t, batch, n_epochs=epochs, lam_k=lam_h, center_res=True, a_value=0.0)

#     E0 = ENet(dz, dx); optE0 = optim.Adam(E0.parameters(), lr=1e-3)  # lam_k=0.0
#     E1 = ENet(dz, dx); optE1 = optim.Adam(E1.parameters(), lr=1e-3)  # lam_k=1.0
#     train_e(E0, optE0, X_t, W_t, Z_t, A_t, batch, n_epochs=epochs, lam_k=0.0, center_res=True)
#     train_e(E1, optE1, X_t, W_t, Z_t, A_t, batch, n_epochs=epochs, lam_k=1.0, center_res=True)

#     with torch.no_grad():
#         wx = torch.cat([W_t, X_t], dim=1)
#         zx = torch.cat([Z_t, X_t], dim=1)
#         h1 = H1(wx); h0 = H0(wx)
#         e0 = E0(zx).clamp_(clamp_eps, 1.0-clamp_eps)
#         e1 = E1(zx).clamp_(clamp_eps, 1.0-clamp_eps)

#         wA0, w0A0 = A_t/e0, (1.0-A_t)/(1.0-e0)
#         wA1, w0A1 = A_t/e1, (1.0-A_t)/(1.0-e1)

#         DR     = torch.mean(h1 - h0 + wA0*(Y_t - h1) - w0A0*(Y_t - h0)).item()
#         DR_sta = torch.mean(h1 - h0 + wA1*(Y_t - h1) - w0A1*(Y_t - h0)).item()
#     return DR, DR_sta


当前使用的方案 4:
Z_cols = ['paco21', 'ph1']
W_cols = ['pafi1', 'hema1']
Loaded RHC: n=5735, dimX=70, dimZ=2, dimW=2
[train_e_stable] epoch 5, loss=0.0334266
[train_e_stable] epoch 10, loss=0.0329692
[train_e_stable] epoch 15, loss=0.0329904
[train_e_stable] epoch 20, loss=0.0332598
[train_e_stable] epoch 25, loss=0.0330611
[train_e_stable] epoch 30, loss=0.0329271
[train_e_stable] epoch 35, loss=0.0331289
[train_e_stable] epoch 40, loss=0.0332564
[train_e_stable] epoch 45, loss=0.033111
[train_e_stable] epoch 50, loss=0.032739
[train_e_stable] epoch 55, loss=0.0333162
[train_e_stable] epoch 60, loss=0.0330435
[train_e_stable] epoch 65, loss=0.0331571
[train_e_stable] epoch 70, loss=0.0330569
[train_e_stable] epoch 75, loss=0.0331
[train_e_stable] epoch 80, loss=0.0331968
[train_e_stable] epoch 85, loss=0.0331829
[train_e_stable] epoch 90, loss=0.0330541
[train_e_stable] epoch 95, loss=0.0335176
[train_e_stable] epoch 100, loss=0.0330825
[train_e_stable] epoch 105, loss=0.0332616
[train_e_sta

In [3]:
# ---------------- 6) Plug-in (Influence-Function) SEs & 95% CIs ----------------
# 说明：
#  - 不再做bootstrap，不做任何重复训练，直接用训练好的 (H1, H0, E_dr, E_sta) 计算影响函数
#  - 区间构造：CI = point ± 1.96 * sd(phi)/sqrt(n)
#  - 为了与原文和你的打印格式一致，保持两套：DR 与 DR(sta)
print(torch.mean(h1 - h0).item())
print(torch.mean(h1_sta - h0_sta).item())
print(torch.mean(  wA_dr*(y_all ) -  w0A_dr*(y_all)).item())
print(torch.mean(  wA_st*(y_all ) -  w0A_st*(y_all)).item())
DR_point     = torch.mean(h1 - h0 + wA_dr*(y_all - h1) - w0A_dr*(y_all - h0)).item()
DR_sta_point = torch.mean(h1_sta - h0_sta + wA_st*(y_all - h1_sta) - w0A_st*(y_all - h0_sta)).item()
print(DR_point)
print(DR_sta_point)
# print(torch.mean( wA1*(Y_t ) - w0A1*(Y_t )).item())
#补充一个Linear function的baseline:

with torch.no_grad():
    # 逐样本的 nuisances
    wx = torch.cat([w_all, x_all], dim=1)
    zx = torch.cat([z_all, x_all], dim=1)
    h1 = H1(wx)                                  # h1(W,X)
    h0 = H0(wx)                                  # h0(W,X)
    h1_sta = H1_sta(wx)                                  # h1(W,X)
    h0_sta = H0_sta(wx)                                  # h0(W,X)
    e_dr  = E_dr(zx).clamp_(clamp_eps, 1.0-clamp_eps)
    e_sta = E_sta(zx).clamp_(clamp_eps, 1.0-clamp_eps)

    # 单样本打分（phi），与点估计公式一一对应
    mu_diff = (h1 - h0)
    mu_diff_sta = (h1_sta - h0_sta)

    phi_dr  = mu_diff + (a_all / e_dr)  * (y_all - h1) \
                        - ((1.0 - a_all) / (1.0 - e_dr)) * (y_all - h0)

    phi_drs = mu_diff_sta + (a_all / e_sta) * (y_all - h1) \
                        - ((1.0 - a_all) / (1.0 - e_sta)) * (y_all - h0)

    # 影响函数中心化（可选，方差不变；这里用中心化写法更贴合理论）
    psi_dr  = (phi_dr  - DR_point).squeeze()
    psi_drs = (phi_drs - DR_sta_point).squeeze()

    # 标准误：sd(psi)/sqrt(n)；unbiased=True 等价于 ddof=1
    se_dr  = float(torch.std(psi_dr,  unbiased=True) / (n ** 0.5))
    se_drs = float(torch.std(psi_drs, unbiased=True) / (n ** 0.5))

    # 95% 正态近似置信区间
    ci_dr  = (DR_point      - 1.96 * se_dr,  DR_point      + 1.96 * se_dr)
    ci_drs = (DR_sta_point  - 1.96 * se_drs, DR_sta_point  + 1.96 * se_drs)

# ---------------- 7) Print like the paper ----------------
print("\n================ RHC: Average Treatment Effect ================")
print(f"DR        : {DR_point:+.4f} ({se_dr:.5f})")
print(f"95% CIs   : [{ci_dr[0]:+.4f}, {ci_dr[1]:+.4f}]")
print("---------------------------------------------------------------")
print(f"DR(sta)   : {DR_sta_point:+.4f} ({se_drs:.5f})")
print(f"95% CIs   : [{ci_drs[0]:+.4f}, {ci_drs[1]:+.4f}]")
print("===============================================================")


-0.04432414472103119
-0.0017395152244716883
-0.027841875329613686
-0.3706607222557068
-0.04470248892903328
-0.022995365783572197

================ RHC: Average Treatment Effect ================
DR        : -0.0447 (0.00606)
95% CIs   : [-0.0566, -0.0328]
---------------------------------------------------------------
DR(sta)   : -0.0230 (0.00596)
95% CIs   : [-0.0347, -0.0113]


In [4]:
################检查我们的方法随着深度和光度变化的稳定性#############################






In [1]:
# ---------------- 6.5) Linear-Closed baseline (closed-form bridge; ATE, plug-in SE) ----------------
# 公式： J_hat = E_n[Tpsi]^T · (E_n[phi psi^T])^+ · E_n[Y phi]
# plug-in 方差：以 REG 形式的逐样本 g_i = (Tpsi_i)^T theta_h 构造 sd(g_i - mean)/sqrt(n)

import numpy as np


import numpy as np, pandas as pd
import torch, torch.nn as nn, torch.nn.functional as F, torch.optim as optim
from math import sqrt

# ---------------- 1) Load & preprocess ----------------
url = "https://hbiostat.org/data/repo/rhc.csv"   # Vanderbilt Biostatistics RHC
df = pd.read_csv(url)

# A = RHC 指示；Y = 30-day survival (和论文表方向一致)
A = (df["swang1"].astype(str).str.upper().str.strip() == "RHC").astype(np.float32).values[:, None]
Y = (df["dth30"].astype(str).str.upper().str.strip() == "NO").astype(np.float32).values[:, None]

# 负控变量（与论文设置一致）
Z_cols = ["pafi1", "paco21"]   # NC exposures
W_cols = ["ph1",   "hema1"]    # NC outcomes





from itertools import combinations
# 所有变量
all_cols = ["pafi1", "paco21", "ph1", "hema1"]
# 生成所有 6 种分配方案：从 4 个里选 2 个做 Z，其余做 W
schemes = []
for z_pair in combinations(all_cols, 2):
    w_pair = [col for col in all_cols if col not in z_pair]
    schemes.append((list(z_pair), w_pair))
# === 这里用一个参数来控制采用哪一种方案（1~6）===
scheme_id = 4   # 改成 1,2,3,4,5,6 之一
if not (1 <= scheme_id <= len(schemes)):
    raise ValueError(f"scheme_id 必须在 1~{len(schemes)} 之间")
Z_cols, W_cols = schemes[scheme_id - 1]
print(f"当前使用的方案 {scheme_id}:")
print("Z_cols =", Z_cols)
print("W_cols =", W_cols)





for c in Z_cols + W_cols:
    if c not in df.columns:
        raise ValueError(f"Missing column {c} in RHC csv")

# X = baseline 协变量（除去 A、Y、W、Z 及明显非特征列）
drop_cols = ["ptid","sadmdte","dschdte","dthdte","lstctdte","t3d30","swang1","dth30"] + Z_cols + W_cols
X_df = df.drop(columns=[c for c in drop_cols if c in df.columns], errors="ignore")

# One-hot（drop_first 以避免共线），均值填补 + 标准化
num_cols = X_df.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = [c for c in X_df.columns if c not in num_cols]
X = pd.get_dummies(X_df, columns=cat_cols, drop_first=True)
X = X.replace([np.inf, -np.inf], np.nan).fillna(X.mean()).astype(np.float32)
X = ((X - X.mean())/X.std(ddof=0)).fillna(0.0)

# W, Z 简单均值填补 + 标准化
def stdize(M):
    M = M.astype(np.float32).copy()
    m = np.nanmean(M, axis=0); M[np.isnan(M)] = np.take(m, np.where(np.isnan(M))[1])
    M -= M.mean(0); M /= (M.std(0, ddof=0) + 1e-8)
    return M

Z = stdize(df[Z_cols].values)
W = stdize(df[W_cols].values)

X = X.values.astype(np.float32)
A = A.astype(np.float32); Y = Y.astype(np.float32)

n, dx = X.shape
dz, dw = Z.shape[1], W.shape[1]
print(f"Loaded RHC: n={n}, dimX={dx}, dimZ={dz}, dimW={dw}")







# ---------------- 6.5) Linear-Closed baseline (closed-form bridge; ATE, plug-in SE) ----------------
# 公式： J_hat = E_n[Tpsi]^T · (E_n[phi psi^T])^+ · E_n[Y phi]
# plug-in 方差：以 REG 形式的逐样本 g_i = (Tpsi_i)^T theta_h 构造 sd(g_i - mean)/sqrt(n)

import numpy as np

# 使用 numpy.float64 提高数值稳定性
X_np = X.astype(np.float64)
Z_np = Z.astype(np.float64)
W_np = W.astype(np.float64)
A_np = A.astype(np.float64)
Y_np = Y.astype(np.float64)

n = X_np.shape[0]
ones = np.ones((n, 1), dtype=np.float64)

# 线性基
b_zx = np.concatenate([ones, Z_np, X_np], axis=1)  # n x d1
b_wx = np.concatenate([ones, W_np, X_np], axis=1)  # n x d2
d1, d2 = b_zx.shape[1], b_wx.shape[1]

# 分块特征（phi, psi）
phi_lin = np.concatenate([(1.0 - A_np) * b_zx, A_np * b_zx], axis=1)   # n x (2*d1)
psi_lin = np.concatenate([(1.0 - A_np) * b_wx, A_np * b_wx], axis=1)   # n x (2*d2)

# Tpsi（与 A 无关）：二元 ATE 下相当于 “(a=1) − (a=0)”
Tpsi_lin = np.concatenate([-b_wx, +b_wx], axis=1)                      # n x (2*d2)

# 样本矩
M_hat = (phi_lin.T @ psi_lin) / n                     # (2*d1) x (2*d2)
v_hat = (phi_lin * Y_np).mean(axis=0)                 # (2*d1,)
c_hat = Tpsi_lin.mean(axis=0)                         # (2*d2,)

# 闭式求解
M_pinv   = np.linalg.pinv(M_hat, rcond=1e-8)          # (2*d2) x (2*d1)
theta_h  = M_pinv @ v_hat                             # (2*d2,) —— h 的线性系数
g_per    = Tpsi_lin @ theta_h                         # (n,)    —— 逐样本贡献 (Th)(W,X)

LC_point = float(g_per.mean())
psi_lc   = g_per - LC_point
LC_se    = float(np.std(psi_lc, ddof=1) / np.sqrt(n))
LC_ci    = (LC_point - 1.96 * LC_se, LC_point + 1.96 * LC_se)

print("---------------------------------------------------------------")
print(f"Linear-closed : {LC_point:+.4f} ({LC_se:.5f})")
print(f"95% CIs       : [{LC_ci[0]:+.4f}, {LC_ci[1]:+.4f}]")
print("===============================================================")


##########################################################################
###########################################################################

#######do interaction
import numpy as np

# ---------- inputs ----------
X_np = X.astype(np.float64)
Z_np = Z.astype(np.float64)
W_np = W.astype(np.float64)
A_np = A.astype(np.float64).reshape(-1, 1)
Y_np = Y.astype(np.float64).reshape(-1, 1)

n, dx = X_np.shape
dz = Z_np.shape[1]
dw = W_np.shape[1]
ones = np.ones((n, 1), dtype=np.float64)

# =========================================================
# (NEW) 0) 选 X 中与 Y 最相关的 top-2 列（按 |corr|）
# =========================================================
top_k = 2
y = Y_np.reshape(-1)

# center
Xc = X_np - X_np.mean(axis=0, keepdims=True)
yc = y - y.mean()

stdX = Xc.std(axis=0, ddof=0)
stdY = yc.std(ddof=0)

# 防止除 0
den = (stdX * stdY) + 1e-12

# Pearson corr = cov / (stdX*stdY)
corr = (Xc * yc[:, None]).mean(axis=0) / den
abs_corr = np.abs(corr)

top_idx = np.argsort(-abs_corr)[:top_k]     # indices of top-k
X_top = X_np[:, top_idx]                   # (n, top_k)

print("Top-k X dims by |corr(X_j, Y)|:")
for rank, j in enumerate(top_idx, start=1):
    print(f"  rank {rank}: X[:, {j}]  corr={corr[j]:+.6f}  |corr|={abs_corr[j]:.6f}")

# =========================================================
# (NEW) 1) 只用 X_top 生成 interaction：zx_top / wx_top
# =========================================================
# zx_top: vec(Z * X_top^T) -> (n, dz*top_k)
zx_top = (Z_np[:, :, None] * X_top[:, None, :]).reshape(n, dz * top_k)

# wx_top: vec(W * X_top^T) -> (n, dw*top_k)
wx_top = (W_np[:, :, None] * X_top[:, None, :]).reshape(n, dw * top_k)

# =========================================================
# 2) 线性基：phi(z,x,zx_top), psi(w,x,wx_top)
#    这里我默认保留 X 的全量主效应，只缩减 interaction
# =========================================================
b_zx = np.concatenate([ones, Z_np, X_np, zx_top], axis=1)   # (n, d1)
b_wx = np.concatenate([ones, W_np, X_np, wx_top], axis=1)   # (n, d2)

d1 = b_zx.shape[1]
d2 = b_wx.shape[1]
print(f"[aug basis] d1={d1} (=1+dz+dx+dz*top_k), d2={d2} (=1+dw+dx+dw*top_k)")

# =========================================================
# 3) 分块特征（phi, psi）
# =========================================================
phi_lin = np.concatenate([(1.0 - A_np) * b_zx, A_np * b_zx], axis=1)   # (n, 2*d1)
psi_lin = np.concatenate([(1.0 - A_np) * b_wx, A_np * b_wx], axis=1)   # (n, 2*d2)

# =========================================================
# 4) Tpsi：ATE 的差分算子 (a=1)-(a=0)
# =========================================================
Tpsi_lin = np.concatenate([-b_wx, +b_wx], axis=1)  # (n, 2*d2)

# (如果要估计 Y(1)/Y(0) 单臂均值，用下面替换)
# Tpsi_lin = np.concatenate([np.zeros_like(b_wx), b_wx], axis=1)   # Y(1)
# Tpsi_lin = np.concatenate([b_wx, np.zeros_like(b_wx)], axis=1)   # Y(0)

# =========================================================
# 5) 样本矩 + 闭式解
# =========================================================
M_hat = (phi_lin.T @ psi_lin) / n
v_hat = (phi_lin * Y_np).mean(axis=0)
c_hat = Tpsi_lin.mean(axis=0)

M_pinv  = np.linalg.pinv(M_hat, rcond=1e-8)
theta_h = M_pinv @ v_hat
g_per   = Tpsi_lin @ theta_h

LC_point = float(g_per.mean())
psi_lc   = g_per - LC_point
LC_se    = float(np.std(psi_lc, ddof=1) / np.sqrt(n))
LC_ci    = (LC_point - 1.96 * LC_se, LC_point + 1.96 * LC_se)

print("---------------------------------------------------------------")
print(f"Linear-closed (+top2-interaction): {LC_point:+.6f} ({LC_se:.6f})")
print(f"95% CIs                          : [{LC_ci[0]:+.6f}, {LC_ci[1]:+.6f}]")
print("===============================================================")

# =========================================================
# 6) sanity：验证 Th == h1 - h0
# =========================================================
theta0 = theta_h[:d2]
theta1 = theta_h[d2:]
h0 = b_wx @ theta0
h1 = b_wx @ theta1
max_err = float(np.max(np.abs(g_per - (h1 - h0))))
print(f"[sanity] max |Th - (h1 - h0)| = {max_err:.3e}")







##########################################################################
###########################################################################
#######do quar

import numpy as np

# ---------- inputs ----------
X_np = X.astype(np.float64)
Z_np = Z.astype(np.float64)
W_np = W.astype(np.float64)
A_np = A.astype(np.float64).reshape(-1, 1)
Y_np = Y.astype(np.float64).reshape(-1, 1)

n, dx = X_np.shape
dz = Z_np.shape[1]
dw = W_np.shape[1]
ones = np.ones((n, 1), dtype=np.float64)

# =========================================================
# (NEW) 0) 选 X 中与 Y 最相关的 top-2 列（按 |corr|）
# =========================================================
top_k = 2
y = Y_np.reshape(-1)

# center
Xc = X_np - X_np.mean(axis=0, keepdims=True)
yc = y - y.mean()

stdX = Xc.std(axis=0, ddof=0)
stdY = yc.std(ddof=0)

# 防止除 0
den = (stdX * stdY) + 1e-12

# Pearson corr = cov / (stdX*stdY)
corr = (Xc * yc[:, None]).mean(axis=0) / den
abs_corr = np.abs(corr)

top_idx = np.argsort(-abs_corr)[:top_k]     # indices of top-k
X_top = X_np[:, top_idx]                   # (n, top_k)

print("Top-k X dims by |corr(X_j, Y)|:")
for rank, j in enumerate(top_idx, start=1):
    print(f"  rank {rank}: X[:, {j}]  corr={corr[j]:+.6f}  |corr|={abs_corr[j]:.6f}")

# =========================================================
# (NEW) 1) 只用 X_top 生成 interaction：zx_top / wx_top
# =========================================================
# zx_top: vec(Z * X_top^T) -> (n, dz*top_k)
zx_top = (Z_np[:, :, None] * X_top[:, None, :]).reshape(n, dz * top_k)

# wx_top: vec(W * X_top^T) -> (n, dw*top_k)
wx_top = (W_np[:, :, None] * X_top[:, None, :]).reshape(n, dw * top_k)

# =========================================================
# 2) 线性基：phi(z,x,zx_top), psi(w,x,wx_top)
#    这里我默认保留 X 的全量主效应，只缩减 interaction
# =========================================================
b_zx = np.concatenate([ones, Z_np, X_np, Z_np**2], axis=1)   # (n, d1)
b_wx = np.concatenate([ones, W_np, X_np, W_np**2], axis=1)   # (n, d2)

d1 = b_zx.shape[1]
d2 = b_wx.shape[1]
# print(f"[aug basis] d1={d1} (=1+dz+dx+dz*top_k), d2={d2} (=1+dw+dx+dw*top_k)")

# =========================================================
# 3) 分块特征（phi, psi）
# =========================================================
phi_lin = np.concatenate([(1.0 - A_np) * b_zx, A_np * b_zx], axis=1)   # (n, 2*d1)
psi_lin = np.concatenate([(1.0 - A_np) * b_wx, A_np * b_wx], axis=1)   # (n, 2*d2)

# =========================================================
# 4) Tpsi：ATE 的差分算子 (a=1)-(a=0)
# =========================================================
Tpsi_lin = np.concatenate([-b_wx, +b_wx], axis=1)  # (n, 2*d2)

# (如果要估计 Y(1)/Y(0) 单臂均值，用下面替换)
# Tpsi_lin = np.concatenate([np.zeros_like(b_wx), b_wx], axis=1)   # Y(1)
# Tpsi_lin = np.concatenate([b_wx, np.zeros_like(b_wx)], axis=1)   # Y(0)

# =========================================================
# 5) 样本矩 + 闭式解
# =========================================================
M_hat = (phi_lin.T @ psi_lin) / n
v_hat = (phi_lin * Y_np).mean(axis=0)
c_hat = Tpsi_lin.mean(axis=0)

M_pinv  = np.linalg.pinv(M_hat, rcond=1e-8)
theta_h = M_pinv @ v_hat
g_per   = Tpsi_lin @ theta_h

LC_point = float(g_per.mean())
psi_lc   = g_per - LC_point
LC_se    = float(np.std(psi_lc, ddof=1) / np.sqrt(n))
LC_ci    = (LC_point - 1.96 * LC_se, LC_point + 1.96 * LC_se)

print("---------------------------------------------------------------")
print(f"Linear-closed (+top2-quar): {LC_point:+.6f} ({LC_se:.6f})")
print(f"95% CIs                          : [{LC_ci[0]:+.6f}, {LC_ci[1]:+.6f}]")
print("===============================================================")

# =========================================================
# 6) sanity：验证 Th == h1 - h0
# =========================================================
theta0 = theta_h[:d2]
theta1 = theta_h[d2:]
h0 = b_wx @ theta0
h1 = b_wx @ theta1
max_err = float(np.max(np.abs(g_per - (h1 - h0))))
print(f"[sanity] max |Th - (h1 - h0)| = {max_err:.3e}")





############################################################
######do iteraction+quar

#######do quar
#######do interaction
import numpy as np

# ---------- inputs ----------
X_np = X.astype(np.float64)
Z_np = Z.astype(np.float64)
W_np = W.astype(np.float64)
A_np = A.astype(np.float64).reshape(-1, 1)
Y_np = Y.astype(np.float64).reshape(-1, 1)

n, dx = X_np.shape
dz = Z_np.shape[1]
dw = W_np.shape[1]
ones = np.ones((n, 1), dtype=np.float64)

# =========================================================
# (NEW) 0) 选 X 中与 Y 最相关的 top-2 列（按 |corr|）
# =========================================================
top_k = 2
y = Y_np.reshape(-1)

# center
Xc = X_np - X_np.mean(axis=0, keepdims=True)
yc = y - y.mean()

stdX = Xc.std(axis=0, ddof=0)
stdY = yc.std(ddof=0)

# 防止除 0
den = (stdX * stdY) + 1e-12

# Pearson corr = cov / (stdX*stdY)
corr = (Xc * yc[:, None]).mean(axis=0) / den
abs_corr = np.abs(corr)

top_idx = np.argsort(-abs_corr)[:top_k]     # indices of top-k
X_top = X_np[:, top_idx]                   # (n, top_k)

print("Top-k X dims by |corr(X_j, Y)|:")
for rank, j in enumerate(top_idx, start=1):
    print(f"  rank {rank}: X[:, {j}]  corr={corr[j]:+.6f}  |corr|={abs_corr[j]:.6f}")

# =========================================================
# (NEW) 1) 只用 X_top 生成 interaction：zx_top / wx_top
# =========================================================
# zx_top: vec(Z * X_top^T) -> (n, dz*top_k)
zx_top = (Z_np[:, :, None] * X_top[:, None, :]).reshape(n, dz * top_k)

# wx_top: vec(W * X_top^T) -> (n, dw*top_k)
wx_top = (W_np[:, :, None] * X_top[:, None, :]).reshape(n, dw * top_k)

# =========================================================
# 2) 线性基：phi(z,x,zx_top), psi(w,x,wx_top)
#    这里我默认保留 X 的全量主效应，只缩减 interaction
# =========================================================
b_zx = np.concatenate([ones, Z_np, X_np, zx_top, Z_np**2], axis=1)   # (n, d1)
b_wx = np.concatenate([ones, W_np, X_np, wx_top, W_np**2], axis=1)   # (n, d2)

d1 = b_zx.shape[1]
d2 = b_wx.shape[1]
# print(f"[aug basis] d1={d1} (=1+dz+dx+dz*top_k), d2={d2} (=1+dw+dx+dw*top_k)")

# =========================================================
# 3) 分块特征（phi, psi）
# =========================================================
phi_lin = np.concatenate([(1.0 - A_np) * b_zx, A_np * b_zx], axis=1)   # (n, 2*d1)
psi_lin = np.concatenate([(1.0 - A_np) * b_wx, A_np * b_wx], axis=1)   # (n, 2*d2)

# =========================================================
# 4) Tpsi：ATE 的差分算子 (a=1)-(a=0)
# =========================================================
Tpsi_lin = np.concatenate([-b_wx, +b_wx], axis=1)  # (n, 2*d2)

# (如果要估计 Y(1)/Y(0) 单臂均值，用下面替换)
# Tpsi_lin = np.concatenate([np.zeros_like(b_wx), b_wx], axis=1)   # Y(1)
# Tpsi_lin = np.concatenate([b_wx, np.zeros_like(b_wx)], axis=1)   # Y(0)

# =========================================================
# 5) 样本矩 + 闭式解
# =========================================================
M_hat = (phi_lin.T @ psi_lin) / n
v_hat = (phi_lin * Y_np).mean(axis=0)
c_hat = Tpsi_lin.mean(axis=0)

M_pinv  = np.linalg.pinv(M_hat, rcond=1e-8)
theta_h = M_pinv @ v_hat
g_per   = Tpsi_lin @ theta_h

LC_point = float(g_per.mean())
psi_lc   = g_per - LC_point
LC_se    = float(np.std(psi_lc, ddof=1) / np.sqrt(n))
LC_ci    = (LC_point - 1.96 * LC_se, LC_point + 1.96 * LC_se)

print("---------------------------------------------------------------")
print(f"Linear-closed (+top2-interaction+quar): {LC_point:+.6f} ({LC_se:.6f})")
print(f"95% CIs                          : [{LC_ci[0]:+.6f}, {LC_ci[1]:+.6f}]")
print("===============================================================")

# =========================================================
# 6) sanity：验证 Th == h1 - h0
# =========================================================
theta0 = theta_h[:d2]
theta1 = theta_h[d2:]
h0 = b_wx @ theta0
h1 = b_wx @ theta1
max_err = float(np.max(np.abs(g_per - (h1 - h0))))
print(f"[sanity] max |Th - (h1 - h0)| = {max_err:.3e}")














当前使用的方案 4:
Z_cols = ['paco21', 'ph1']
W_cols = ['pafi1', 'hema1']
Loaded RHC: n=5735, dimX=70, dimZ=2, dimW=2
---------------------------------------------------------------
Linear-closed : +0.9134 (0.05901)
95% CIs       : [+0.7978, +1.0291]
Top-k X dims by |corr(X_j, Y)|:
  rank 1: X[:, 47]  corr=-0.521311  |corr|=0.521311
  rank 2: X[:, 15]  corr=+0.390143  |corr|=0.390143
[aug basis] d1=77 (=1+dz+dx+dz*top_k), d2=77 (=1+dw+dx+dw*top_k)
---------------------------------------------------------------
Linear-closed (+top2-interaction): -0.106840 (0.396469)
95% CIs                          : [-0.883918, +0.670239]
[sanity] max |Th - (h1 - h0)| = 5.684e-14
Top-k X dims by |corr(X_j, Y)|:
  rank 1: X[:, 47]  corr=-0.521311  |corr|=0.521311
  rank 2: X[:, 15]  corr=+0.390143  |corr|=0.390143
---------------------------------------------------------------
Linear-closed (+top2-quar): -0.528468 (0.183271)
95% CIs                          : [-0.887679, -0.169258]
[sanity] max |Th - (h1 - h0)|